## DFW Commercial Asset — Distributed Energy Potential Study

## Description
This notebook implements an automated, reproducible pipeline to identify, enrich, spatially analyze, and model the rooftop solar energy generation potential of large commercial and industrial buildings across the Dallas-Fort Worth metroplex.

DATA SOURCE  : OpenStreetMap via OSMnx  
STUDY AREA   : 11 DFW counties  
THRESHOLD    : Buildings > 20,000 sq ft


## Essential library installation

In [ ]:
!pip install osmnx geopandas pandas numpy requests folium branca pydeck keplergl matplotlib seaborn scipy --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.4/18.4 MB 36.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 19.8 MB/s eta 0:00:00


## Importing Libraries

In [ ]:
import os
import math
import warnings
import requests
import numpy as np
import pandas as pd
import geopandas as gpd

import folium
from folium import plugins
from branca.colormap import LinearColormap

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats
from shapely.geometry import Point, Polygon, MultiPolygon

warnings.filterwarnings('ignore')  # suppress non-critical warnings for clean output

# Confirm versions
print("pandas     :", pd.__version__)
print("geopandas  :", gpd.__version__)
print("numpy      :", np.__version__)
print("folium     :", folium.__version__)
print("Environment ready.")

pandas     : 2.2.2
geopandas  : 1.1.3
numpy      : 2.0.2
folium     : 0.20.0
Environment ready.


## Global Configuration


In [ ]:

# Study area
STUDY_AREA_NAME = "DFW Metroplex, Texas"

# CRS settings
# EPSG:4326 = WGS84 lat/lon (GPS coordinates, standard for web maps)
# EPSG:2276 = Texas North Central (feet) for accurate area math
CRS_GEOGRAPHIC = "EPSG:4326"
CRS_PROJECTED  = "EPSG:2276"   # NAD83 / Texas North Central, units = US survey feet

# Building filter threshold
MIN_BUILDING_SQ_FT = 20000

# Roof area assumption
# Usable roof area is not 100% of the building footprint.
# HVAC units, skylights, parapets, and setback requirements
# typically reduce usable solar area to ~75–80%.
ROOF_USABLE_FRACTION = 0.65   # 80% of Building_Sq_Ft = Roof_Sq_Ft

# Solar energy model constants (DFW-specific)
# These values come from NREL's National Solar Radiation Database (NSRDB)
# and standard photovoltaic engineering references.
PANEL_EFFICIENCY   = 0.20     # 20% — standard commercial-grade monocrystalline panel
PERFORMANCE_RATIO  = 0.80     # 80% — accounts for inverter loss, wiring, shading, temp
SOLAR_HOURS_PER_DAY = 5.5     # Peak sun hours/day for Dallas-Fort Worth (NREL NSRDB)
DAYS_PER_YEAR       = 365

# Derived constant: annual solar radiation on a tilted surface (kWh/m²/year)
# H = peak_sun_hours × days_per_year
ANNUAL_SOLAR_RADIATION = SOLAR_HOURS_PER_DAY * DAYS_PER_YEAR   # = 2007.5 kWh/m²/year

# Unit conversion
# Building_Sq_Ft is in US survey feet. Solar formula uses square meters.
SQ_FT_TO_SQ_M = 0.0929       # 1 square foot = 0.0929 square meters

# Household energy benchmark
# EIA (US Energy Information Administration) reports average Texas household
# uses approximately 1,096 kWh/month = 13,152 kWh/year (2024 data)
AVG_HOUSEHOLD_KWH_PER_YEAR = 13152

# Buffer distances for spatial analysis
BUFFER_1_MILE_METERS = 1609.34
BUFFER_3_MILE_METERS = 4828.03

# --- Output folder ---
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")
print(f"Annual solar radiation (H): {ANNUAL_SOLAR_RADIATION} kWh/m²/year")

Output directory ready: /content/outputs
Annual solar radiation (H): 2007.5 kWh/m²/year


## Data Cleaning and Enrichment

In [ ]:
from google.colab import files

print("Please upload your CSV file...")
uploaded = files.upload()

# Get the filename dynamically (works regardless of what you named the file)
csv_filename = list(uploaded.keys())[0]
print(f"\nFile uploaded: {csv_filename}")

# Load into a pandas DataFrame
df = pd.read_csv(csv_filename)

print(f"\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nColumn names and data types:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

Please upload your CSV file...


Saving DFW_Metroplex_Commercial_Assets_20k.csv to DFW_Metroplex_Commercial_Assets_20k.csv

File uploaded: DFW_Metroplex_Commercial_Assets_20k.csv

Dataset shape: 8612 rows × 6 columns

Column names and data types:
name               object
building           object
Building_Sq_Ft    float64
county             object
latitude          float64
longitude         float64
dtype: object

First 5 rows:


,name,building,Building_Sq_Ft,county,latitude,longitude
0,NorthPark Center,retail,1.219857e+06,"Dallas County, Texas",32.868393,-96.773500
1,NaN,industrial,1.206589e+06,"Dallas County, Texas",32.717254,-97.030512
2,NaN,commercial,5.527943e+04,"Dallas County, Texas",32.979636,-96.715736
3,Unbridled Living of Dallas,commercial,1.151029e+05,"Dallas County, Texas",32.916452,-96.770310
4,NaN,office,5.662391e+04,"Dallas County, Texas",32.923485,-96.785250


## Data Quality Audit

In [ ]:
print("=" * 55)
print("DATA QUALITY AUDIT")
print("=" * 55)

print(f"\nTotal records     : {len(df):,}")
print(f"Total columns     : {df.shape[1]}")

print("\n--- Missing values per column ---")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
audit = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(audit)

print("\n--- Building type distribution ---")
print(df['building'].value_counts())

print("\n--- County distribution ---")
print(df['county'].value_counts())

print("\n--- Building_Sq_Ft statistics ---")
print(df['Building_Sq_Ft'].describe().apply(lambda x: f"{x:,.0f}"))

print("\n--- Outlier check (buildings > 1,000,000 sq ft) ---")
giants = df[df['Building_Sq_Ft'] > 1_000_000]
print(f"Count: {len(giants)}")
print(giants[['name','building','Building_Sq_Ft','county']].to_string())

DATA QUALITY AUDIT

Total records     : 8,612
Total columns     : 6

--- Missing values per column ---
                Missing Count  Missing %
name                     7125      82.73
building                    0       0.00
Building_Sq_Ft              0       0.00
county                      0       0.00
latitude                    0       0.00
longitude                   0       0.00

--- Building type distribution ---
building
commercial     2562
retail         2260
warehouse      1929
industrial     1283
office          576
supermarket       2
Name: count, dtype: int64

--- County distribution ---
county
Dallas County, Texas      5066
Tarrant County, Texas     1498
Collin County, Texas      1003
Denton County, Texas       738
Ellis County, Texas         86
Johnson County, Texas       66
Rockwall County, Texas      57
Parker County, Texas        29
Kaufman County, Texas       29
Hunt County, Texas          28
Wise County, Texas          12
Name: count, dtype: int64

--- Building_Sq

## Adding Roof_Sq_Ft column

In [ ]:
df['Roof_Sq_Ft'] = (df['Building_Sq_Ft'] * ROOF_USABLE_FRACTION).round(2)

print("Roof_Sq_Ft column added.")
print(f"\nUsable fraction applied: {ROOF_USABLE_FRACTION * 100:.0f}%")
print(f"\nRoof_Sq_Ft statistics:")
print(df['Roof_Sq_Ft'].describe().apply(lambda x: f"{x:,.0f}"))
print("\nSample (first 5 rows):")
df[['name', 'building', 'Building_Sq_Ft', 'Roof_Sq_Ft']].head()

Roof_Sq_Ft column added.

Usable fraction applied: 65%

Roof_Sq_Ft statistics:
count        8,612
mean        61,117
std        107,268
min         13,003
25%         17,790
50%         27,538
75%         53,164
max      3,254,244
Name: Roof_Sq_Ft, dtype: object

Sample (first 5 rows):


,name,building,Building_Sq_Ft,Roof_Sq_Ft
0,NorthPark Center,retail,1.219857e+06,792906.86
1,NaN,industrial,1.206589e+06,784282.60
2,NaN,commercial,5.527943e+04,35931.63
3,Unbridled Living of Dallas,commercial,1.151029e+05,74816.88
4,NaN,office,5.662391e+04,36805.54


## Loading previously extracted nearest street file

In [ ]:
print('Loading the previously extracted data from street_checkpoint.csv...')
df = pd.read_csv('/content/street_checkpoint.csv')

print(f"\nDataset shape after loading checkpoint: {df.shape[0]} rows \u00d7 {df.shape[1]} columns")
print("\nFirst 5 rows of the loaded DataFrame with 'Nearest_Street':")
display(df.head())

Loading the previously extracted data from street_checkpoint.csv...

Dataset shape after loading checkpoint: 8612 rows × 9 columns

First 5 rows of the loaded DataFrame with 'Nearest_Street':


,name,building,Building_Sq_Ft,county,latitude,longitude,Roof_Sq_Ft,Nearest_Street,street_done
0,NorthPark Center,retail,1.219857e+06,"Dallas County, Texas",32.868393,-96.773500,975885.37,North Central Expressway,1
1,NaN,industrial,1.206589e+06,"Dallas County, Texas",32.717254,-97.030512,965270.89,Lance Drive,1
2,NaN,commercial,5.527943e+04,"Dallas County, Texas",32.979636,-96.715736,44223.54,North Central Expressway,1
3,Unbridled Living of Dallas,commercial,1.151029e+05,"Dallas County, Texas",32.916452,-96.770310,92082.31,Coit Road,1
4,NaN,office,5.662391e+04,"Dallas County, Texas",32.923485,-96.785250,45299.13,Hillcrest Road,1


## Fill missing building names

In [ ]:
# PHASE 2 — CELL 8: Fill missing building names
# 82.8% of records have no name in OSM. We attempt to fill
# them using Nominatim's display_name field.
# For unnamed buildings, we generate a descriptive label:
#   "[Building Type] on [Street Name]"
# This gives every record a human-readable identifier.

def get_display_name(lat, lon):
    """
    Queries Nominatim for a human-readable location name.
    Returns the amenity/shop/office name if found, else None.
    """
    try:
        params = {
            'lat': lat,
            'lon': lon,
            'format': 'json',
            'addressdetails': 1
        }
        headers = {'User-Agent': USER_AGENT}
        response = requests.get(NOMINATIM_URL, params=params, headers=headers, timeout=10)
        if response.status_code == 200:
            data = response.json()
            address = data.get('address', {})
            name = (
                address.get('amenity') or
                address.get('shop') or
                address.get('office') or
                address.get('building') or
                None
            )
            return name
    except:
        return None
    return None

# Only process rows where name is missing to avoid redundant API calls
missing_name_mask = df['name'].isna()
print(f"Rows with missing name: {missing_name_mask.sum():,}")

# For speed in this cell, we use a fallback label approach first
# (API-based name filling is done exactly like Cell 7 above)
# Fallback label = "[Building type] on [Street]"
df['name'] = df.apply(
    lambda row: row['name'] if pd.notna(row['name'])
    else f"{str(row['building']).capitalize()} on {row.get('Nearest_Street', 'Unknown Street')}",
    axis=1
)

print(f"\nAll names now filled. Sample of generated names:")
print(df[missing_name_mask]['name'].head(10).to_string())

Rows with missing name: 7,125

All names now filled. Sample of generated names:
1                  Industrial on Lance Drive
2     Commercial on North Central Expressway
4                   Office on Hillcrest Road
5             Commercial on River Bend Drive
6         Commercial on Empire Central Drive
7               Industrial on Joe Field Road
8                 Warehouse on Maybank Drive
9                  Warehouse on Denton Drive
10            Commercial on East 10th Street
11             Commercial on Cockrell Avenue


In [ ]:
# ============================================================
# PHASE 2 — CELL 9: Finalize and save the enriched dataset
# ============================================================

# Define the final column order for clarity
final_columns = [
    'name',
    'building',
    'county',
    'Nearest_Street',
    'latitude',
    'longitude',
    'Building_Sq_Ft',
    'Roof_Sq_Ft',
]

df = df[final_columns].copy()

# Final audit
print("ENRICHED DATASET SUMMARY")
print("=" * 45)
print(f"Total records   : {len(df):,}")
print(f"Columns         : {list(df.columns)}")
print(f"Missing values  :\n{df.isnull().sum()}")

# Save to CSV
enriched_path = f"{OUTPUT_DIR}/DFW_Assets_Enriched.csv"
df.to_csv(enriched_path, index=False)
print(f"\nSaved to: {enriched_path}")
df.head()

ENRICHED DATASET SUMMARY
Total records   : 8,612
Columns         : ['name', 'building', 'county', 'Nearest_Street', 'latitude', 'longitude', 'Building_Sq_Ft', 'Roof_Sq_Ft']
Missing values  :
name                0
building            0
county              0
Nearest_Street    112
latitude            0
longitude           0
Building_Sq_Ft      0
Roof_Sq_Ft          0
dtype: int64

Saved to: /content/outputs/DFW_Assets_Enriched.csv


,name,building,county,Nearest_Street,latitude,longitude,Building_Sq_Ft,Roof_Sq_Ft
0,NorthPark Center,retail,"Dallas County, Texas",North Central Expressway,32.868393,-96.773500,1.219857e+06,975885.37
1,Industrial on Lance Drive,industrial,"Dallas County, Texas",Lance Drive,32.717254,-97.030512,1.206589e+06,965270.89
2,Commercial on North Central Expressway,commercial,"Dallas County, Texas",North Central Expressway,32.979636,-96.715736,5.527943e+04,44223.54
3,Unbridled Living of Dallas,commercial,"Dallas County, Texas",Coit Road,32.916452,-96.770310,1.151029e+05,92082.31
4,Office on Hillcrest Road,office,"Dallas County, Texas",Hillcrest Road,32.923485,-96.785250,5.662391e+04,45299.13


## Handling missing values in 'Nearest_Street' Column

In [ ]:
# Fill any remaining missing values in 'Nearest_Street' with 'Unknown Street'
print(f"Missing 'Nearest_Street' values before filling: {df['Nearest_Street'].isnull().sum()}")
df['Nearest_Street'] = df['Nearest_Street'].fillna('Unknown Street')
print(f"Missing 'Nearest_Street' values after filling: {df['Nearest_Street'].isnull().sum()}")

display(df[df['Nearest_Street'] == 'Unknown Street'].head())

Missing 'Nearest_Street' values before filling: 112
Missing 'Nearest_Street' values after filling: 0


,name,building,county,Nearest_Street,latitude,longitude,Building_Sq_Ft,Roof_Sq_Ft
8500,Tractor Supply Company,retail,"Kaufman County, Texas",Unknown Street,32.739231,-96.301444,95891.319812,76713.06
8501,Retail on nan,retail,"Kaufman County, Texas",Unknown Street,32.742221,-96.278917,39023.286614,31218.63
8502,"Dal-Bac Manufacturing Company, Inc.",warehouse,"Kaufman County, Texas",Unknown Street,32.735980,-96.443920,49327.447853,39461.96
8503,Retail on nan,retail,"Kaufman County, Texas",Unknown Street,32.718577,-96.326893,33461.103355,26768.88
8504,Retail on nan,retail,"Kaufman County, Texas",Unknown Street,32.719656,-96.326213,33620.633563,26896.51


## Spatial Analysis

In [ ]:
# PHASE 3: Convert to GeoDataFrame

# We convert our pandas DataFrame into a GeoPandas GeoDataFrame.
# Each row gets a Point geometry built from its lat/lon.
# We set the CRS to WGS84 (EPSG:4326) first, then reproject
# to EPSG:2276 (Texas North Central, feet) for all spatial math.

# Build Point geometry from lat/lon columns
geometry = [Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])]

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_GEOGRAPHIC)

print(f"GeoDataFrame created: {len(gdf):,} rows")
print(f"CRS: {gdf.crs}")

# Reproject to Texas North Central (feet) for accurate distance/area calculations
gdf_proj = gdf.to_crs(CRS_PROJECTED)
print(f"\nReprojected CRS: {gdf_proj.crs}")
print("Units: US survey feet — buffer distances must be in feet.")

# Convert buffer distances from meters to feet for EPSG:2276
BUFFER_1_MILE_FEET = BUFFER_1_MILE_METERS * 3.28084   # 5,280 ft
BUFFER_3_MILE_FEET = BUFFER_3_MILE_METERS * 3.28084   # 15,840 ft

print(f"\n1-mile buffer = {BUFFER_1_MILE_FEET:,.0f} ft")
print(f"3-mile buffer = {BUFFER_3_MILE_FEET:,.0f} ft")

GeoDataFrame created: 8,612 rows
CRS: EPSG:4326

Reprojected CRS: EPSG:2276
Units: US survey feet — buffer distances must be in feet.

1-mile buffer = 5,280 ft
3-mile buffer = 15,840 ft


##  Fetching Census household data via API

In [ ]:
# ============================================================
# PHASE 3 — CELL 11: Fetch Census household data
# ============================================================
# US Census Bureau now requires a free API key for all requests.
# Get yours (free, instant) at: https://api.census.gov/data/key_signup.html
# Paste it below or store it in Colab Secrets as CENSUS_API_KEY
# ============================================================

import requests
import pandas as pd
import time

# ── API Key ──
# OPTION A (recommended): store in Colab Secrets
# Click the lock icon in the left sidebar → Add new secret
# Name it exactly: CENSUS_API_KEY
try:
    from google.colab import userdata
    CENSUS_API_KEY = userdata.get('CENSUS_API_KEY')
    print("Census API key loaded from Colab Secrets.")
except Exception:
    # OPTION B: paste key directly — remove before sharing notebook
    CENSUS_API_KEY = "YOUR_CENSUS_API_KEY_HERE"
    print("WARNING: API key hardcoded. Remove before sharing this notebook.")

CENSUS_API = "https://api.census.gov/data/2024/acs/acs5"

DFW_COUNTY_FIPS = {
    "Dallas"   : "48113",
    "Tarrant"  : "48439",
    "Collin"   : "48085",
    "Denton"   : "48121",
    "Ellis"    : "48139",
    "Johnson"  : "48251",
    "Parker"   : "48367",
    "Kaufman"  : "48257",
    "Rockwall" : "48397",
    "Hunt"     : "48231",
    "Wise"     : "48497",
}

def fetch_census_tracts(state_fips, county_fips, max_retries=3):
    """
    Fetches total household count per census tract for one county.

    Parameters:
        state_fips  (str): 2-digit state FIPS code ('48' for Texas)
        county_fips (str): Full 5-digit FIPS e.g. '48113'
        max_retries (int): Retry attempts on failure

    Returns:
        pd.DataFrame: columns = full_geoid, households, county_fips, NAME
    """
    county_3digit = county_fips[2:]   # '48113' → '113'

    params = {
        'get' : 'B11001_001E,NAME',
        'for' : 'tract:*',
        'in'  : f'state:{state_fips} county:{county_3digit}',
        'key' : 'b16b889a6de224f22e372c4089e9304e67175993'           # ← the only change from before
    }

    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(CENSUS_API, params=params, timeout=30)

            # Check HTTP status
            if r.status_code != 200:
                print(f"  HTTP {r.status_code} for {county_fips}. "
                      f"Response: {r.text[:200]}")
                time.sleep(2 * attempt)
                continue

            # Check we got JSON and not an HTML error page
            content_type = r.headers.get('Content-Type', '')
            if 'html' in content_type.lower():
                print(f"  Got HTML response (possible auth error):")
                print(f"  {r.text[:300]}")
                break   # no point retrying an auth error

            # Parse JSON
            try:
                data = r.json()
            except Exception:
                print(f"  JSON parse failed. Raw: {r.text[:300]}")
                time.sleep(2 * attempt)
                continue

            if len(data) < 2:
                print(f"  Empty data for county {county_fips}")
                return pd.DataFrame()

            # Build DataFrame
            col_headers = data[0]
            rows        = data[1:]
            df_c = pd.DataFrame(rows, columns=col_headers)
            df_c.rename(columns={'B11001_001E': 'households'}, inplace=True)
            df_c['households'] = pd.to_numeric(
                df_c['households'], errors='coerce'
            ).fillna(0)

            # 11-digit GEOID: state(2) + county(3) + tract(6)
            # Matches TIGER shapefile GEOID field exactly
            df_c['full_geoid']  = state_fips + county_3digit + df_c['tract']
            df_c['county_fips'] = county_fips

            return df_c[['full_geoid', 'households', 'county_fips', 'NAME']]

        except requests.exceptions.Timeout:
            print(f"  Timeout attempt {attempt}/{max_retries}. Retrying...")
            time.sleep(3 * attempt)

        except Exception as e:
            print(f"  Error for {county_fips}: {e}")
            break

    return pd.DataFrame()


# ── Fetch for all 11 counties ──
all_census = []
print("Fetching Census household data for all DFW counties...\n")

for county_name, fips in DFW_COUNTY_FIPS.items():
    print(f"Fetching: {county_name} County ({fips})...")
    result = fetch_census_tracts("48", fips)

    if not result.empty:
        print(f"  {len(result):,} tracts — "
              f"{result['households'].sum():,.0f} households")
        all_census.append(result)
    else:
        print(f"  WARNING: No data for {county_name} County")

    time.sleep(0.3)   # polite delay between counties

# ── Combine ──
if not all_census:
    print("\nERROR: No data fetched. Check your API key is correct.")
else:
    census_df = pd.concat(all_census, ignore_index=True)
    print(f"\n{'='*50}")
    print(f"CENSUS DATA FETCHED SUCCESSFULLY")
    print(f"{'='*50}")
    print(f"Total census tracts : {len(census_df):,}")
    print(f"Total households    : {census_df['households'].sum():,.0f}")

    print(f"\nHouseholds by county:")
    county_name_map = {v: k for k, v in DFW_COUNTY_FIPS.items()}
    for fips, hh in census_df.groupby('county_fips')['households'].sum().items():
        print(f"  {county_name_map.get(fips, fips):<12} : {hh:>10,.0f}")

    print(f"\nSample rows:")
    print(census_df.head(5).to_string())

Fetching Census household data for all DFW counties...

Fetching: Dallas County (48113)...
  645 tracts — 982,737 households
Fetching: Tarrant County (48439)...
  449 tracts — 782,419 households
Fetching: Collin County (48085)...
  220 tracts — 416,646 households
Fetching: Denton County (48121)...
  193 tracts — 360,392 households
Fetching: Ellis County (48139)...
  36 tracts — 71,759 households
Fetching: Johnson County (48251)...
  39 tracts — 65,507 households
Fetching: Parker County (48367)...
  29 tracts — 56,437 households
Fetching: Kaufman County (48257)...
  27 tracts — 50,990 households
Fetching: Rockwall County (48397)...
  29 tracts — 41,237 households
Fetching: Hunt County (48231)...
  21 tracts — 38,466 households
Fetching: Wise County (48497)...
  16 tracts — 25,135 households

CENSUS DATA FETCHED SUCCESSFULLY
Total census tracts : 1,704
Total households    : 2,891,725

Households by county:
  Collin       :    416,646
  Dallas       :    982,737
  Denton       :    360,39

In [ ]:
census_df.to_csv("DFW_Census_Households.csv", index=False)

In [ ]:
# ============================================================
# PHASE 3 — CELL 12: Download Census tract shapefiles (fixed)
# ============================================================
# FIX: Census TIGER/Line 2025 no longer provides individual
# county-level tract files. We download the full Texas state
# tract file and filter to DFW counties afterward.
#
# State-level file URL format:
#   https://www2.census.gov/geo/tiger/TIGER2025/TRACT/tl_2025_48_tract.zip
#   where 48 = Texas FIPS code
# ============================================================

import urllib.request
import zipfile
import os
import geopandas as gpd
import pandas as pd

STATE_FIPS  = "48"   # Texas
TIGER_BASE  = "https://www2.census.gov/geo/tiger/TIGER2025/TRACT"
filename    = f"tl_2025_{STATE_FIPS}_tract.zip"
url         = f"{TIGER_BASE}/{filename}"
local_zip   = f"/tmp/{filename}"
extract_dir = f"/tmp/tx_tracts"

# ── Step 1: Download the full Texas tract shapefile ──
print(f"Downloading Texas state tract shapefile...")
print(f"URL: {url}")
print(f"This is one file (~15MB) covering all Texas census tracts.")
print(f"Please wait...", end=" ")

try:
    urllib.request.urlretrieve(url, local_zip)
    print("Downloaded.")
except Exception as e:
    print(f"\nDownload failed: {e}")
    print("Try opening this URL in your browser to verify it exists:")
    print(url)
    raise

# ── Step 2: Extract the zip file ──
print("Extracting...", end=" ")
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(local_zip, 'r') as z:
    z.extractall(extract_dir)
print("Done.")

# List extracted files so we can confirm the .shp is there
extracted_files = os.listdir(extract_dir)
print(f"Extracted files: {extracted_files}")

# ── Step 3: Load the shapefile into a GeoDataFrame ──
shp_file = [f for f in extracted_files if f.endswith('.shp')][0]
print(f"\nLoading shapefile: {shp_file}...")

tx_tracts_gdf = gpd.read_file(f"{extract_dir}/{shp_file}")
print(f"Total Texas census tracts loaded: {len(tx_tracts_gdf):,}")
print(f"Columns: {list(tx_tracts_gdf.columns)}")
print(f"CRS: {tx_tracts_gdf.crs}")

# ── Step 4: Filter to DFW counties only ──
# The GEOID field in the shapefile is an 11-digit string:
#   state(2) + county(3) + tract(6)
# The COUNTYFP field is just the 3-digit county portion.
# We extract it and keep only the 11 DFW counties.

# Build a set of 3-digit county FIPS codes for our 11 counties
dfw_county_3digit = {fips[2:] for fips in DFW_COUNTY_FIPS.values()}
# e.g. {'113', '439', '085', '121', '139', '251', '367', '257', '397', '231', '497'}

print(f"\nFiltering to DFW counties: {dfw_county_3digit}")

tracts_gdf = tx_tracts_gdf[
    tx_tracts_gdf['COUNTYFP'].isin(dfw_county_3digit)
].copy()

print(f"DFW tract polygons after filter: {len(tracts_gdf):,}")

# Add readable county name for reference
fips_to_name = {fips[2:]: name for name, fips in DFW_COUNTY_FIPS.items()}
tracts_gdf['county_name'] = tracts_gdf['COUNTYFP'].map(fips_to_name)

# ── Step 5: Reproject to Texas North Central (feet) ──
# Required for accurate buffer distance calculations later
tracts_gdf = tracts_gdf.to_crs(CRS_PROJECTED)
print(f"Reprojected to: {tracts_gdf.crs}")

# ── Step 6: Merge household counts from census_df ──
# Match on the 11-digit GEOID
tracts_gdf['GEOID']            = tracts_gdf['GEOID'].astype(str).str.strip()
census_df['full_geoid']        = census_df['full_geoid'].astype(str).str.strip()

tracts_gdf = tracts_gdf.merge(
    census_df[['full_geoid', 'households']],
    left_on  = 'GEOID',
    right_on = 'full_geoid',
    how      = 'left'   # keep all tracts even if household count is missing
)

# ── Step 7: Results summary ──
matched   = tracts_gdf['households'].notna().sum()
unmatched = tracts_gdf['households'].isna().sum()

print(f"\n{'='*50}")
print(f"TRACT SHAPEFILE READY")
print(f"{'='*50}")
print(f"Total DFW tract polygons      : {len(tracts_gdf):,}")
print(f"Tracts with household data    : {matched:,}")
print(f"Tracts WITHOUT household data : {unmatched:,}")

if unmatched > 0:
    print(f"\nWARNING: {unmatched} tracts did not match census data.")
    print("This usually means a GEOID format mismatch.")
    print("Showing unmatched GEOIDs for inspection:")
    print(tracts_gdf[tracts_gdf['households'].isna()]['GEOID'].head(5).tolist())
    print("Showing census_df full_geoid sample:")
    print(census_df['full_geoid'].head(5).tolist())

print(f"\nTotal households in study area: "
      f"{tracts_gdf['households'].sum():,.0f}")
print(f"\nTracts per county:")
print(tracts_gdf.groupby('county_name')['households'].agg(['count','sum'])
      .rename(columns={'count':'tracts','sum':'households'})
      .sort_values('households', ascending=False)
      .to_string())

URL: https://www2.census.gov/geo/tiger/TIGER2025/TRACT/tl_2025_48_tract.zip
This is one file (~15MB) covering all Texas census tracts.
Please wait... Downloaded.
Extracting... Done.
Extracted files: ['tl_2025_48_tract.shx', 'tl_2025_48_tract.dbf', 'tl_2025_48_tract.shp.ea.iso.xml', 'tl_2025_48_tract.shp', 'tl_2025_48_tract.shp.iso.xml', 'tl_2025_48_tract.prj', 'tl_2025_48_tract.cpg']

Loading shapefile: tl_2025_48_tract.shp...
Total Texas census tracts loaded: 6,896
Columns: ['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']
CRS: EPSG:4269

Filtering to DFW counties: {'121', '231', '251', '113', '497', '085', '257', '367', '139', '439', '397'}
DFW tract polygons after filter: 1,704
Reprojected to: EPSG:2276

TRACT SHAPEFILE READY
Total DFW tract polygons      : 1,704
Tracts with household data    : 1,704
Tracts WITHOUT household data : 0

Total households in study area: 2,891,725

Tracts

##  Buffer analysis and household count per building
This is the core spatial analysis. For every building we create a 1-mile and 3-mile buffer, then count how many households fall inside.

In [ ]:
# ============================================================
# PHASE 3 — CELL 13: Buffer analysis — households per building
# ============================================================
# For each building:
#   1. Create a 1-mile buffer polygon around the building's point
#   2. Find all census tracts that intersect that buffer
#   3. Calculate the proportion of each tract inside the buffer
#   4. Multiply that proportion by the tract's household count
#   5. Sum → total estimated households within 1 mile
# Repeat for 3-mile buffer.
# This is called "areal interpolation" and is standard GIS practice.
# ============================================================

def count_households_in_buffer(building_point, buffer_distance_ft, tracts_gdf):
    """
    Estimates the number of households within a buffer around a building.
    Uses areal interpolation: weighted by the fraction of each tract's
    area that falls inside the buffer.

    Parameters:
        building_point    : Shapely Point (in EPSG:2276 feet)
        buffer_distance_ft: Buffer radius in feet
        tracts_gdf        : GeoDataFrame of census tracts (in EPSG:2276)

    Returns:
        int: Estimated households in buffer
    """
    buffer_poly = building_point.buffer(buffer_distance_ft)

    # Find tracts that intersect the buffer
    intersecting = tracts_gdf[tracts_gdf.intersects(buffer_poly)].copy()

    if intersecting.empty:
        return 0

    # Calculate intersection area for each tract
    intersecting['intersection_area'] = intersecting.geometry.intersection(buffer_poly).area
    intersecting['tract_area']        = intersecting.geometry.area

    # Proportion of each tract that falls inside the buffer
    intersecting['proportion'] = (
        intersecting['intersection_area'] / intersecting['tract_area']
    ).clip(0, 1)

    # Weighted household count
    intersecting['hh_in_buffer'] = intersecting['proportion'] * intersecting['households'].fillna(0)

    return int(intersecting['hh_in_buffer'].sum())

# --- Apply to all buildings ---
# Note: This loop may take 20–40 minutes for 8,612 buildings.
# A progress counter is printed every 500 rows.

print("Calculating household counts for all buildings...")
print("This will take approximately 20–40 minutes. Please wait.\n")

hh_1mile = []
hh_3mile = []

for i, row in gdf_proj.iterrows():
    pt = row.geometry
    hh_1mile.append(count_households_in_buffer(pt, BUFFER_1_MILE_FEET, tracts_gdf))
    hh_3mile.append(count_households_in_buffer(pt, BUFFER_3_MILE_FEET, tracts_gdf))

    if (i + 1) % 500 == 0:
        print(f"  Processed {i+1:,} / {len(gdf_proj):,} buildings...")

gdf_proj['Households_1mi'] = hh_1mile
gdf_proj['Households_3mi'] = hh_3mile

print("\nHousehold analysis complete.")
print(f"\nAvg households within 1 mile: {gdf_proj['Households_1mi'].mean():,.0f}")
print(f"Avg households within 3 miles: {gdf_proj['Households_3mi'].mean():,.0f}")

Calculating household counts for all buildings...
This will take approximately 20–40 minutes. Please wait.

  Processed 500 / 8,612 buildings...
  Processed 1,000 / 8,612 buildings...
  Processed 1,500 / 8,612 buildings...
  Processed 2,000 / 8,612 buildings...
  Processed 2,500 / 8,612 buildings...
  Processed 3,000 / 8,612 buildings...
  Processed 3,500 / 8,612 buildings...
  Processed 4,000 / 8,612 buildings...
  Processed 4,500 / 8,612 buildings...
  Processed 5,000 / 8,612 buildings...
  Processed 5,500 / 8,612 buildings...
  Processed 6,000 / 8,612 buildings...
  Processed 6,500 / 8,612 buildings...
  Processed 7,000 / 8,612 buildings...
  Processed 7,500 / 8,612 buildings...
  Processed 8,000 / 8,612 buildings...
  Processed 8,500 / 8,612 buildings...

Household analysis complete.

Avg households within 1 mile: 4,417
Avg households within 3 miles: 39,640


##Fetching NLR EV charging station data

In [ ]:
# ============================================================
# PHASE 3 — CELL 14: Fetch EV charging stations from NREL
# ============================================================
# The NREL Alternative Fuels Station Locator API provides
# locations of all registered EV charging stations in the US.
#
# API documentation:
#   https://developer.nrel.gov/docs/transportation/alt-fuel-stations-v1/
#
# HOW TO GET YOUR FREE API KEY:
#   1. Go to https://developer.nrel.gov/signup/
#   2. Register for a free account
#   3. Copy your API key and store it in Colab Secrets as NREL_API_KEY
#
# NOTE ON REBRANDING:
#   NREL (National Renewable Energy Laboratory) was rebranded
#   to National Laboratories of the Rockies but the API
#   endpoint and documentation remain at developer.nrel.gov
# ============================================================

from shapely.geometry import Point
import requests
import pandas as pd
import geopandas as gpd

# ── API Key ──
try:
    from google.colab import userdata
    NREL_API_KEY = userdata.get('NREL_API_KEY')
    print("NREL API key loaded from Colab Secrets.")
except Exception:
    NREL_API_KEY = "a9cydLHTlWbXNp3b9c10XDLYQKgRhshM2teE0rEz"
    print("WARNING: API key hardcoded. Remove before sharing notebook.")

NREL_URL = "https://developer.nrel.gov/api/alt-fuel-stations/v1.json"

# ── Step 1: Diagnostic request first ──
# We test with minimal parameters to confirm the API key works
# before sending the full request.
print("\nRunning API key diagnostic...")

diag_params = {
    'api_key'   : NREL_API_KEY,
    'fuel_type' : 'ELEC',
    'state'     : 'TX',
    'limit'     : 1,          # just 1 result to test
    'status'    : 'E',
}

diag = requests.get(NREL_URL, params=diag_params, timeout=30)
print(f"Diagnostic status code: {diag.status_code}")

if diag.status_code == 401:
    print("ERROR: Invalid API key. Check your key at developer.nrel.gov")
    raise ValueError("Invalid NREL API key")
elif diag.status_code == 422:
    print(f"422 detail: {diag.text[:500]}")
    raise ValueError("Parameter error — see detail above")
elif diag.status_code != 200:
    print(f"Unexpected error: {diag.text[:300]}")
    raise ValueError(f"API returned {diag.status_code}")
else:
    print("API key confirmed working.\n")

# ── Step 2: Full request — DFW bounding box ──
# Instead of fetching all of Texas (10,000+ stations) and
# filtering afterward, we use a bounding box around DFW to
# get only relevant stations. This is more efficient and
# avoids the limit parameter issues.
#
# DFW bounding box (approximate):
#   Latitude  : 32.0 to 33.6 (south to north)
#   Longitude : -98.2 to -96.0 (west to east)

print("Fetching EV stations within DFW bounding box...")

full_params = {
    'api_key'    : NREL_API_KEY,
    'fuel_type'  : 'ELEC',
    'status'     : 'E',          # E = currently open/available
    'bbox'       : '32.0,-98.2,33.6,-96.0',  # lat_min,lon_min,lat_max,lon_max
}

response = requests.get(NREL_URL, params=full_params, timeout=60)
print(f"Response status: {response.status_code}")

if response.status_code != 200:
    print(f"Error detail: {response.text[:500]}")

    # ── Fallback: fetch by state without bbox ──
    # Some API versions don't support bbox — try state-only as fallback
    print("\nTrying fallback: state-level fetch without bbox...")
    fallback_params = {
        'api_key'   : NREL_API_KEY,
        'fuel_type' : 'ELEC',
        'state'     : 'TX',
        'status'    : 'E',
    }
    response = requests.get(NREL_URL, params=fallback_params, timeout=60)
    print(f"Fallback status: {response.status_code}")

if response.status_code == 200:
    ev_data     = response.json()
    ev_stations = pd.DataFrame(ev_data['fuel_stations'])

    print(f"Raw stations fetched: {len(ev_stations):,}")
    print(f"Columns available: {list(ev_stations.columns)}")

    # ── Step 3: Keep relevant columns ──
    # We only need location + charger count information
    ev_cols = [
        'id',
        'station_name',
        'street_address',
        'city',
        'state',
        'zip',
        'latitude',
        'longitude',
        'ev_level1_evse_num',    # Level 1 chargers (slowest)
        'ev_level2_evse_num',    # Level 2 chargers (standard)
        'ev_dc_fast_num',        # DC fast chargers (fastest)
    ]
    # Only keep columns that actually exist in the response
    ev_cols_present = [c for c in ev_cols if c in ev_stations.columns]
    ev_stations     = ev_stations[ev_cols_present].copy()

    # ── Step 4: Drop rows with missing coordinates ──
    before = len(ev_stations)
    ev_stations = ev_stations.dropna(subset=['latitude', 'longitude']).copy()
    after  = len(ev_stations)
    print(f"Dropped {before - after} rows with missing coordinates.")

    # ── Step 5: Filter to DFW bounding box if we got statewide data ──
    # If the bbox parameter wasn't supported and we got all of Texas,
    # filter down to DFW manually here.
    dfw_mask = (
        (ev_stations['latitude']  >= 32.0) &
        (ev_stations['latitude']  <= 33.6) &
        (ev_stations['longitude'] >= -98.2) &
        (ev_stations['longitude'] <= -96.0)
    )
    ev_stations_dfw = ev_stations[dfw_mask].copy()
    print(f"EV stations within DFW bounding box: {len(ev_stations_dfw):,}")

    # ── Step 6: Build total charger count column ──
    # Sum all charger types per station for a single count metric
    for col in ['ev_level1_evse_num', 'ev_level2_evse_num', 'ev_dc_fast_num']:
        if col in ev_stations_dfw.columns:
            ev_stations_dfw[col] = pd.to_numeric(
                ev_stations_dfw[col], errors='coerce'
            ).fillna(0)

    charger_cols = [c for c in
                    ['ev_level1_evse_num','ev_level2_evse_num','ev_dc_fast_num']
                    if c in ev_stations_dfw.columns]

    ev_stations_dfw['total_chargers'] = ev_stations_dfw[charger_cols].sum(axis=1)

    # ── Step 7: Convert to GeoDataFrame ──
    ev_geometry = [
        Point(row['longitude'], row['latitude'])
        for _, row in ev_stations_dfw.iterrows()
    ]

    ev_gdf = gpd.GeoDataFrame(
        ev_stations_dfw,
        geometry=ev_geometry,
        crs=CRS_GEOGRAPHIC       # WGS84 lat/lon
    )

    # Reproject to Texas North Central (feet) for spatial joins
    ev_gdf_proj = ev_gdf.to_crs(CRS_PROJECTED)

    # ── Step 8: Save to CSV ──
    ev_stations_dfw.to_csv(f"{OUTPUT_DIR}/EV_Stations_DFW.csv", index=False)

    # ── Summary ──
    print(f"\n{'='*50}")
    print(f"EV STATIONS READY")
    print(f"{'='*50}")
    print(f"Total DFW EV stations        : {len(ev_gdf_proj):,}")
    print(f"Total charger ports (approx) : "
          f"{ev_stations_dfw['total_chargers'].sum():,.0f}")
    print(f"\nCharger type breakdown:")
    for col in charger_cols:
        label = col.replace('ev_','').replace('_evse_num','').replace('_',' ')
        print(f"  {label:<20}: "
              f"{ev_stations_dfw[col].sum():,.0f} ports")
    print(f"\nTop 10 cities by station count:")
    if 'city' in ev_stations_dfw.columns:
        print(ev_stations_dfw['city'].value_counts().head(10).to_string())
    print(f"\nSaved to: {OUTPUT_DIR}/EV_Stations_DFW.csv")

else:
    print(f"\nAll attempts failed. Status: {response.status_code}")
    print(f"Detail: {response.text[:400]}")
    print("\nCreating empty fallback GeoDataFrame so pipeline can continue...")
    ev_gdf_proj = gpd.GeoDataFrame(
        columns=['station_name','latitude','longitude','total_chargers'],
        geometry=[],
        crs=CRS_PROJECTED
    )
    print("Pipeline will continue with 0 EV stations.")
    print("You can re-run this cell later once the API issue is resolved.")


Running API key diagnostic...
Diagnostic status code: 200
API key confirmed working.

Fetching EV stations within DFW bounding box...
Response status: 200
Raw stations fetched: 85,724
Columns available: ['access_code', 'access_days_time', 'access_detail_code', 'cards_accepted', 'date_last_confirmed', 'expected_date', 'fuel_type_code', 'groups_with_access_code', 'id', 'maximum_vehicle_class', 'open_date', 'owner_type_code', 'related_stations', 'restricted_access', 'status_code', 'funding_sources', 'facility_type', 'station_name', 'station_phone', 'updated_at', 'geocode_status', 'latitude', 'longitude', 'city', 'country', 'intersection_directions', 'plus4', 'state', 'street_address', 'zip', 'bd_blends', 'cng_dispenser_num', 'cng_fill_type_code', 'cng_has_rng', 'cng_psi', 'cng_renewable_source', 'cng_total_compression', 'cng_total_storage', 'cng_vehicle_class', 'e85_blender_pump', 'e85_other_ethanol_blends', 'ev_connector_types', 'ev_dc_fast_num', 'ev_level1_evse_num', 'ev_level2_evse_nu

## Counting EV station per buffer

In [ ]:
# ============================================================
# PHASE 3 — CELL 15: Count EV stations per building buffer
# ============================================================
# For each building, count how many existing EV stations are
# already within the 1-mile and 3-mile buffer zones.
# This tells us which areas are underserved (few EV stations
# but many households = high priority for new infrastructure).
# ============================================================

def count_ev_in_buffer(building_point, buffer_distance_ft, ev_gdf_proj):
    """
    Counts EV charging stations within a buffer distance.
    Uses a spatial index for efficiency on large datasets.

    Parameters:
        building_point    : Shapely Point (EPSG:2276)
        buffer_distance_ft: Buffer radius in feet
        ev_gdf_proj       : GeoDataFrame of EV stations (EPSG:2276)

    Returns:
        int: Number of EV stations within buffer
    """
    if ev_gdf_proj.empty:
        return 0
    buffer_poly = building_point.buffer(buffer_distance_ft)
    return int(ev_gdf_proj[ev_gdf_proj.within(buffer_poly)].shape[0])

print("Counting EV stations per building buffer...")

ev_1mile = []
ev_3mile = []

for i, row in gdf_proj.iterrows():
    pt = row.geometry
    ev_1mile.append(count_ev_in_buffer(pt, BUFFER_1_MILE_FEET, ev_gdf_proj))
    ev_3mile.append(count_ev_in_buffer(pt, BUFFER_3_MILE_FEET, ev_gdf_proj))

    if (i + 1) % 500 == 0:
        print(f"  Processed {i+1:,} / {len(gdf_proj):,}...")

gdf_proj['EV_Stations_1mi'] = ev_1mile
gdf_proj['EV_Stations_3mi'] = ev_3mile

print("\nEV station count complete.")
print(f"Avg EV stations within 1 mile: {gdf_proj['EV_Stations_1mi'].mean():.2f}")
print(f"Avg EV stations within 3 miles: {gdf_proj['EV_Stations_3mi'].mean():.2f}")

Counting EV stations per building buffer...
  Processed 500 / 8,612...
  Processed 1,000 / 8,612...
  Processed 1,500 / 8,612...
  Processed 2,000 / 8,612...
  Processed 2,500 / 8,612...
  Processed 3,000 / 8,612...
  Processed 3,500 / 8,612...
  Processed 4,000 / 8,612...
  Processed 4,500 / 8,612...
  Processed 5,000 / 8,612...
  Processed 5,500 / 8,612...
  Processed 6,000 / 8,612...
  Processed 6,500 / 8,612...
  Processed 7,000 / 8,612...
  Processed 7,500 / 8,612...
  Processed 8,000 / 8,612...
  Processed 8,500 / 8,612...

EV station count complete.
Avg EV stations within 1 mile: 4.89
Avg EV stations within 3 miles: 27.04


## Energy Modeling
## Solar Energy Generation Calculation

In [ ]:
# --- Step 1: Convert Roof_Sq_Ft to square meters ---
# The formula requires area in m² but our data is in sq ft.
# 1 sq ft = 0.0929 m²
gdf_proj['Roof_Sq_M'] = gdf_proj['Roof_Sq_Ft'] * SQ_FT_TO_SQ_M

# --- Step 2: Apply the energy formula ---
# E = A × r × H × PR
gdf_proj['Annual_kWh'] = (
    gdf_proj['Roof_Sq_M']       # A — panel area in m²
    * PANEL_EFFICIENCY           # r — efficiency fraction
    * ANNUAL_SOLAR_RADIATION     # H — kWh/m²/year for DFW
    * PERFORMANCE_RATIO          # PR — system performance ratio
).round(0)

# --- Step 3: Convert kWh to MWh for readability ---
# 1 MWh = 1,000 kWh
gdf_proj['Annual_MWh'] = (gdf_proj['Annual_kWh'] / 1000).round(2)

# --- Step 4: Households powered per year ---
# How many average Texas households could this roof power for one year?
# Based on EIA: average Texas household uses 14,112 kWh/year
gdf_proj['Households_Powered'] = (
    gdf_proj['Annual_kWh'] / AVG_HOUSEHOLD_KWH_PER_YEAR
).round(1)

# --- Step 5: Daily energy output (for context) ---
gdf_proj['Daily_kWh'] = (gdf_proj['Annual_kWh'] / DAYS_PER_YEAR).round(1)

# --- Step 6: Energy priority score ---
# A composite score combining energy potential and population served.
# Higher score = higher priority for solar investment.
# Score = Annual_MWh × Households_1mi (normalized 0-1)
# This allows ranking buildings by their social impact, not just raw size.
max_mwh = gdf_proj['Annual_MWh'].max()
max_hh  = gdf_proj['Households_1mi'].max() if gdf_proj['Households_1mi'].max() > 0 else 1

gdf_proj['Priority_Score'] = (
    (gdf_proj['Annual_MWh'] / max_mwh) * 0.6 +      # 60% weight on generation potential
    (gdf_proj['Households_1mi'] / max_hh) * 0.4       # 40% weight on nearby population
).round(4)

# --- Results summary ---
print("ENERGY MODELING COMPLETE")
print("=" * 50)
print(f"\nFormula: E = A × r × H × PR")
print(f"  r  = {PANEL_EFFICIENCY} ({PANEL_EFFICIENCY*100:.0f}% efficiency)")
print(f"  H  = {ANNUAL_SOLAR_RADIATION:,} kWh/m²/year")
print(f"  PR = {PERFORMANCE_RATIO} ({PERFORMANCE_RATIO*100:.0f}% performance ratio)")
print(f"\nTotal annual generation potential : {gdf_proj['Annual_MWh'].sum():,.0f} MWh/year")
print(f"Total households that could be powered: {gdf_proj['Households_Powered'].sum():,.0f}")
print(f"\nPer-building statistics (Annual_MWh):")
print(gdf_proj['Annual_MWh'].describe().apply(lambda x: f"{x:,.1f}"))

ENERGY MODELING COMPLETE

Formula: E = A × r × H × PR
  r  = 0.2 (20% efficiency)
  H  = 2,007.5 kWh/m²/year
  PR = 0.8 (80% performance ratio)

Total annual generation potential : 19,330,111 MWh/year
Total households that could be powered: 1,469,744

Per-building statistics (Annual_MWh):
count      8,612.0
mean       2,244.6
std        3,939.5
min          477.5
25%          653.3
50%        1,011.3
75%        1,952.5
max      119,513.8
Name: Annual_MWh, dtype: object


In [ ]:
# ============================================================
# PHASE 4 — CELL 17: Save the complete enriched master dataset
# ============================================================
# This is the single source of truth for all downstream
# visualization and analysis. Every column is documented below.
# ============================================================

# Reproject back to WGS84 for compatibility with web mapping tools
gdf_final = gdf_proj.to_crs(CRS_GEOGRAPHIC)

# Column documentation
column_docs = {
    'name'               : 'Building name (from OSM or generated)',
    'building'           : 'OSM building type tag',
    'county'             : 'Texas county name',
    'Nearest_Street'     : 'Nearest road name (from reverse geocoding)',
    'latitude'           : 'Decimal degrees, WGS84',
    'longitude'          : 'Decimal degrees, WGS84',
    'Building_Sq_Ft'     : 'Building footprint area (sq ft, EPSG:2276)',
    'Roof_Sq_Ft'         : f'Usable roof area = Building_Sq_Ft × {ROOF_USABLE_FRACTION}',
    'Roof_Sq_M'          : 'Roof_Sq_Ft converted to m² for solar formula',
    'Annual_kWh'         : 'Estimated solar generation (kWh/year)',
    'Annual_MWh'         : 'Annual_kWh ÷ 1000',
    'Daily_kWh'          : 'Annual_kWh ÷ 365',
    'Households_Powered' : f'Annual_kWh ÷ {AVG_HOUSEHOLD_KWH_PER_YEAR} (EIA Texas avg)',
    'Households_1mi'     : 'Estimated households within 1-mile buffer',
    'Households_3mi'     : 'Estimated households within 3-mile buffer',
    'EV_Stations_1mi'    : 'Existing EV charging stations within 1 mile (NREL)',
    'EV_Stations_3mi'    : 'Existing EV charging stations within 3 miles (NREL)',
    'Priority_Score'     : 'Composite score: 60% generation + 40% population (0-1)',
}

print("FINAL DATASET COLUMNS AND DESCRIPTIONS")
print("=" * 55)
for col, desc in column_docs.items():
    if col in gdf_final.columns:
        print(f"  {col:<22} : {desc}")

# Save as CSV (no geometry, for spreadsheet use)
csv_cols = [c for c in column_docs.keys() if c in gdf_final.columns]
gdf_final[csv_cols].to_csv(f"{OUTPUT_DIR}/DFW_Assets_Master.csv", index=False)

# Save as GeoJSON (includes geometry, for GIS tools)
gdf_final.to_file(f"{OUTPUT_DIR}/DFW_Assets_Master.geojson", driver='GeoJSON')

print(f"\nSaved CSV    : {OUTPUT_DIR}/DFW_Assets_Master.csv")
print(f"Saved GeoJSON: {OUTPUT_DIR}/DFW_Assets_Master.geojson")
print(f"\nTotal records in master dataset: {len(gdf_final):,}")

FINAL DATASET COLUMNS AND DESCRIPTIONS
  name                   : Building name (from OSM or generated)
  building               : OSM building type tag
  county                 : Texas county name
  Nearest_Street         : Nearest road name (from reverse geocoding)
  latitude               : Decimal degrees, WGS84
  longitude              : Decimal degrees, WGS84
  Building_Sq_Ft         : Building footprint area (sq ft, EPSG:2276)
  Roof_Sq_Ft             : Usable roof area = Building_Sq_Ft × 0.65
  Roof_Sq_M              : Roof_Sq_Ft converted to m² for solar formula
  Annual_kWh             : Estimated solar generation (kWh/year)
  Annual_MWh             : Annual_kWh ÷ 1000
  Daily_kWh              : Annual_kWh ÷ 365
  Households_Powered     : Annual_kWh ÷ 13152 (EIA Texas avg)
  Households_1mi         : Estimated households within 1-mile buffer
  Households_3mi         : Estimated households within 3-mile buffer
  EV_Stations_1mi        : Existing EV charging stations within 1 mi

## Visualization & Interactive Mapping

In [ ]:
import folium
from folium import plugins
import json
import geopandas as gpd

# Work with WGS84 for Folium (Leaflet uses lat/lon)
gdf_map = gdf_final.copy()

# --- Create DFW Metroplex outline and County outlines in WGS84 ---
# tracts_gdf is in CRS_PROJECTED, reproject to WGS84 for Folium
tracts_gdf_wgs84 = tracts_gdf.to_crs(CRS_GEOGRAPHIC)

# Dissolve all DFW tracts to form a single Metroplex boundary
dfw_metroplex_boundary_gdf = tracts_gdf_wgs84.dissolve()
dfw_metroplex_geometry = dfw_metroplex_boundary_gdf.geometry.iloc[0]

# Dissolve tracts by county to get individual county boundaries
dfw_county_boundaries_gdf = tracts_gdf_wgs84.dissolve(by='county_name').reset_index()

# --- Color scheme by building type ---
BUILDING_COLORS = {
    'commercial'  : '#4A90D9',   # blue
    'warehouse'   : '#E67E22',   # orange
    'industrial'  : '#8E44AD',   # purple
    'retail'      : '#27AE60',   # green
    'office'      : '#2C3E50',   # dark navy
    'supermarket' : '#E74C3C',   # red
}

def get_color(building_type):
    return BUILDING_COLORS.get(str(building_type).lower(), '#95A5A6')

# --- DFW center coordinates ---
DFW_CENTER = [32.89, -97.04]

# --- Build the Folium map ---
m = folium.Map(
    location=DFW_CENTER,
    zoom_start=10,
    tiles='CartoDB positron',    # clean light basemap, good for research figures
    control_scale=True
)

# Add additional tile layer options
folium.TileLayer('CartoDB dark_matter', name='Dark basemap').add_to(m)
folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)

# --- Feature groups (layers) ---
# Create a dictionary to hold FeatureGroups for each building type
building_type_layers = {}
# Now, FeatureGroups for building types should be visible in the LayerControl
for b_type, _ in BUILDING_COLORS.items():
    fg = folium.FeatureGroup(name=b_type.capitalize(), show=True) # control=True is default
    building_type_layers[b_type] = fg

# These buffers are controlled by JS, so they shouldn't appear in the top-right LayerControl
layer_buffer_1mi   = folium.FeatureGroup(name='1-mile buffers (selected)', show=False, control=False)
layer_buffer_3mi   = folium.FeatureGroup(name='3-mile buffers (selected)', show=False, control=False)

# These layers *should* appear in the top-right LayerControl
layer_ev_stations  = folium.FeatureGroup(name='EV charging stations', show=False)
layer_dfw_outline  = folium.FeatureGroup(name='DFW Metroplex Outline', show=False)
layer_county_outlines = folium.FeatureGroup(name='DFW County Outlines', show=False)

# --- Add EV stations ---
if not ev_gdf_proj.empty:
    ev_wgs = ev_gdf_proj.to_crs(CRS_GEOGRAPHIC)
    for _, ev_row in ev_wgs.iterrows():
        if pd.notna(ev_row.geometry.y):
            folium.CircleMarker(
                location=[ev_row.geometry.y, ev_row.geometry.x],
                radius=4,
                color='#2ECC71',
                fill=True,
                fill_color='#2ECC71',
                fill_opacity=0.8,
                tooltip=str(ev_row.get('station_name', 'EV Station')),
                popup=folium.Popup(f"""
                    <b>{ev_row.get('station_name','EV Station')}</b><br>
                    {ev_row.get('street_address','')}
                """, max_width=250)
            ).add_to(layer_ev_stations)

# --- Add DFW Metroplex outline to its layer ---
if not dfw_metroplex_geometry.is_empty:
    folium.GeoJson(
        dfw_metroplex_geometry,
        name='DFW Metroplex Outline',
        style_function=lambda x: {
            'fillColor': 'none',
            'color': 'black',
            'weight': 3,
            'dashArray': '5, 5'
        }
    ).add_to(layer_dfw_outline)

# --- Add DFW county outlines to its layer ---
folium.GeoJson(
    dfw_county_boundaries_gdf.to_json(),
    name='DFW County Outlines',
    style_function=lambda x: {
        'fillColor': 'none',
        'color': 'grey',
        'weight': 1,
        'dashArray': '3, 3'
    },
    tooltip=folium.features.GeoJsonTooltip(fields=['county_name'])
).add_to(layer_county_outlines)

# --- Add building markers with rich popups ---
# Each building marker has:
#   - Colored circle (by building type)
#   - Popup with full stats
#   - Buffer drawn on click (via JavaScript)

# Prepare data as a JavaScript object for client-side interaction
buildings_js_data = []

for idx, row in gdf_map.iterrows():
    lat = row['latitude']
    lon = row['longitude']

    if pd.isna(lat) or pd.isna(lon):
        continue

    b_type = str(row.get('building', '')).lower()
    color = get_color(b_type)

    # Build a rich HTML popup
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px; min-width: 280px;">
        <h4 style="margin:0 0 8px 0; color:#2C3E50;">{row.get('name','Unknown')}</h4>
        <hr style="margin:4px 0;">
        <table style="width:100%; border-collapse:collapse;">
            <tr><td><b>Type:</b></td><td>{row.get('building','—')}</td></tr>
            <tr><td><b>County:</b></td><td>{str(row.get('county','')).replace(', Texas','')}</td></tr>
            <tr><td><b>Street:</b></td><td>{row.get('Nearest_Street','—')}</td></tr>
            <tr><td><b>Building Sq Ft:</b></td><td>{row.get('Building_Sq_Ft',0):,.0f}</td></tr>
            <tr><td><b>Roof Sq Ft:</b></td><td>{row.get('Roof_Sq_Ft',0):,.0f}</td></tr>
        </table>
        <hr style="margin:6px 0;">
        <b>Solar Energy Potential</b>
        <table style="width:100%; border-collapse:collapse; margin-top:4px;">
            <tr><td>Annual generation:</td><td><b>{row.get('Annual_MWh',0):,.1f} MWh/yr</b></td></tr>
            <tr><td>Households powered:</td><td><b>{row.get('Households_Powered',0):,.0f}</b></td></tr>
        </table>
        <hr style="margin:6px 0;">
        <b>1-Mile Buffer</b>
        <table style="width:100%; border-collapse:collapse; margin-top:4px;">
            <tr><td>Households:</td><td>{row.get('Households_1mi',0):,}</td></tr>
            <tr><td>EV stations:</td><td>{row.get('EV_Stations_1mi',0)}</td></tr>
        </table>
        <b>3-Mile Buffer</b>
        <table style="width:100%; border-collapse:collapse; margin-top:4px;">
            <tr><td>Households:</td><td>{row.get('Households_3mi',0):,}</td></tr>
            <tr><td>EV stations:</td><td>{row.get('EV_Stations_3mi',0)}</td></tr>
        </table>
        <hr style="margin:6px 0;">
        <span style="color:#888; font-size:11px;">Priority Score: {row.get('Priority_Score',0):.4f}</span>
        <br>
        <button onclick="drawBuffers('{m.get_name()}', {lat},{lon},'1mile')"
            style="margin-top:6px; padding:4px 8px; background:#4A90D9; color:white; border:none; border-radius:4px; cursor:pointer;">
            Show 1-mi buffer
        </button>
        <button onclick="drawBuffers('{m.get_name()}', {lat},{lon},'3mile')"
            style="margin-top:4px; padding:4px 8px; background:#E67E22; color:white; border:none; border-radius:4px; cursor:pointer;">
            Show 3-mi buffer
        </button>
    </div>
    """

    marker = folium.CircleMarker(
        location=[lat, lon],
        radius=max(4, min(12, row.get('Annual_MWh', 0) / 500)),  # size by energy potential
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        weight=0.8,
        tooltip=f"{row.get('name','Building')} — {row.get('Annual_MWh',0):,.0f} MWh/yr",
        popup=folium.Popup(popup_html, max_width=320)
    )
    # Add to the appropriate FeatureGroup
    if b_type in building_type_layers:
        marker.add_to(building_type_layers[b_type])

    # Store data for JavaScript multi-select
    buildings_js_data.append({
        'lat'         : round(lat, 6),
        'lon'         : round(lon, 6),
        'name'        : str(row.get('name','')),
        'building'    : str(row.get('building','')),
        'annual_mwh'  : float(row.get('Annual_MWh', 0)),
        'hh_powered'  : float(row.get('Households_Powered', 0)),
        'hh_1mi'      : int(row.get('Households_1mi', 0)),
        'hh_3mi'      : int(row.get('Households_3mi', 0)),
        'ev_1mi'      : int(row.get('EV_Stations_1mi', 0)),
        'ev_3mi'      : int(row.get('EV_Stations_3mi', 0)),
    })


# Add building type layers to the map so they are managed by LayerControl
for layer_group in building_type_layers.values():
    layer_group.add_to(m)

# Combine all overlay layers for LayerControl
overlay_layers = {
    name: fg for name, fg in building_type_layers.items()
}
overlay_layers['EV charging stations'] = layer_ev_stations
overlay_layers['DFW Metroplex Outline'] = layer_dfw_outline
overlay_layers['DFW County Outlines'] = layer_county_outlines

# Add other layers to map that should be managed by the default LayerControl
layer_ev_stations.add_to(m)
layer_dfw_outline.add_to(m)
layer_county_outlines.add_to(m)

# Layer control (top-right toggle panel)
folium.LayerControl(
    collapsed=False,
    overlay_layers=overlay_layers
).add_to(m)

# --- Legend (static color legend only) ---
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:12px 16px; border-radius:8px;
     border:1px solid #ccc; font-family:Arial; font-size:12px; box-shadow:2px 2px 6px rgba(0,0,0,0.2);">
    <b>Building Type</b><br>
"""
for b_type, color_code in BUILDING_COLORS.items():
    legend_html += f"""
    <span style="color:{color_code};">&#9679;</span> {b_type.capitalize()}<br>
    """
legend_html += """
    <hr style="margin:6px 0;">
    <i>Marker size = energy potential</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# --- JavaScript for dynamic buffer drawing and multi-select ---
# This JavaScript runs in the browser when the map HTML is opened.
# It adds two features:
#   1. drawBuffers(mapName, lat, lon, type) — draws a circle buffer on the map
#   2. Multi-select panel — tracks selected buildings and computes
#      centroid buffer + combined stats when "Analyze" is clicked

js_code = f"""
<script>
var activeBuffers = [];   // track drawn buffer layers for cleanup

function drawBuffers(mapName, lat, lon, bufferType) {{
    var mapObj = window['{m.get_name()}']; // Get the map object dynamically here
    if (!mapObj) {{
        console.error("Map object '", mapName, "' not found.");
        return;
    }}
    activeBuffers.forEach(function(layer) {{ mapObj.removeLayer(layer); }});
    activeBuffers = [];

    var radiusMeters = bufferType === '1mile' ? 1609.34 : 4828.03;
    var color = bufferType === '1mile' ? '#4A90D9' : '#E67E22';
    var label = bufferType === '1mile' ? '1-mile buffer' : '3-mile buffer';

    var circle = L.circle([lat, lon], {{
        radius: radiusMeters,
        color: color,
        fillColor: color,
        fillOpacity: 0.12,
        weight: 2,
        dashArray: '6 4'
    }}).addTo(mapObj); // Add to the dynamically retrieved map object
    circle.bindTooltip(label, {{permanent: false}});
    activeBuffers.push(circle);
    mapObj.fitBounds(circle.getBounds());
}}

// ---- Multi-select feature ----
var selectedBuildings = [];
var allBuildingData = {json.dumps(buildings_js_data[:500])};

function addToSelection(lat, lon, name, mwh, hh1, hh3, ev1, ev3) {{
    selectedBuildings.push({{'lat':lat, 'lon':lon, 'name':name, 'mwh':mwh, 'hh1':hh1, 'hh3':hh3, 'ev1':ev1, 'ev3':ev3}});
    document.getElementById('sel-count').innerText = selectedBuildings.length;
    document.getElementById('multi-panel').style.display = 'block';
}}

function analyzeSelection() {{
    if (selectedBuildings.length === 0) return;

    var centLat = selectedBuildings.reduce((s,b) => s+b.lat, 0) / selectedBuildings.length;
    var centLon = selectedBuildings.reduce((s,b) => s+b.lon, 0) / selectedBuildings.length;

    var totalMwh = selectedBuildings.reduce((s,b) => s+b.mwh, 0);
    var totalHH1  = selectedBuildings.reduce((s,b) => s+b.hh1, 0);
    var totalHH3  = selectedBuildings.reduce((s,b) => s+b.hh3, 0);
    var totalEV1  = selectedBuildings.reduce((s,b) => s+b.ev1, 0);

    var mapObj = window['{m.get_name()}']; // Get map object dynamically here too
    if (!mapObj) {{
        console.error("Map object '", '{m.get_name()}', "' not found for analysis.");
        return;
    }}

    activeBuffers.forEach(function(l) {{ mapObj.removeLayer(l); }});
    activeBuffers = [];

    var buf1 = L.circle([centLat, centLon], {{
        radius: 1609.34, color:'#4A90D9', fillColor:'#4A90D9',
        fillOpacity: 0.10, weight:2,
        dashArray: '6 4'
    }}).addTo(mapObj).bindTooltip('1-mi centroid buffer');

    var buf3 = L.circle([centLat, centLon], {{
        radius: 4828.03, color:'#E67E22', fillColor:'#E67E22',
        fillOpacity: 0.07, weight:2,
        dashArray: '6 4'
    }}).addTo(mapObj).bindTooltip('3-mi centroid buffer');

    activeBuffers.push(buf1, buf3);

    var centMark = L.marker([centLat, centLon]).addTo(mapObj)
        .bindPopup('<b>Selection centroid</b><br>'
            + selectedBuildings.length + ' buildings selected<br>'
            + 'Total generation: <b>' + totalMwh.toFixed(1) + ' Mwh/yr</b><br>'
            + 'Households in 1-mi buffer: <b>' + totalHH1.toLocaleString() + '</b><br>'
            + 'Households in 3-mi buffer: <b>' + totalHH3.toLocaleString() + '</b><br>'+
            'EV stations in 1-mi buffer: <b>' + totalEV1 + '</b>'
        ).openPopup();
    activeBuffers.push(centMark);
    mapObj.setView([centLat, centLon], 12);
}}

function clearSelection() {{
    selectedBuildings = [];
    document.getElementById('sel-count').innerText = '0';
    document.getElementById('multi-panel').style.display = 'none';

    var mapObj = window['{m.get_name()}']; // Get map object dynamically here too
    if (mapObj) {{
        activeBuffers.forEach(function(l) {{ mapObj.removeLayer(l); }});
    }}
    activeBuffers = [];
}}

// Attach the click listener to the map itself using the *name* from Python
// We defer this so the map object is guaranteed to be initialized.
document.addEventListener('DOMContentLoaded', function() {{
    var mapObj = window['{m.get_name()}'];
    if (mapObj) {{
        mapObj.on('click', function(e) {{
            // Ensure the click is on the map pane itself, not on a marker or popup
            if (e.originalEvent.target.classList.contains('leaflet-container')) {{
                clearSelection();
            }}
        }});
    }} else {{
        console.error("Map object '{m.get_name()}' not found for click listener attachment.");
    }}
}});
</script>

<div id="multi-panel" style="display:none; position:fixed; top:80px; right:10px; z-index:2000;
     background:white; padding:12px; border-radius:8px; border:1px solid #ccc;
     font-family:Arial; font-size:12px; max-width:200px; box-shadow:2px 2px 6px rgba(0,0,0,0.2);">
    <b>Multi-select mode</b><br>
    <span id="sel-count">0</span> buildings selected<br>
    <button onclick="analyzeSelection()"
        style="margin-top:8px; width:100%; padding:5px; background:#27AE60; color:white; border:none; border-radius:4px; cursor:pointer;">
        Analyze selection
    </button>
    <button onclick="clearSelection()"
        style="margin-top:4px; width:100%; padding:5px; background:#E74C3C; color:white; border:none; border-radius:4px; cursor:pointer;">
        Clear
    </button>
</div>
"""
m.get_root().html.add_child(folium.Element(js_code))

# Save map
map_path = f"{OUTPUT_DIR}/DFW_Energy_Map.html"
m.save(map_path)
print(f"Interactive map saved: {map_path}")
print("Download this file and open it in any browser to use the map.")

Interactive map saved: /content/outputs/DFW_Energy_Map.html
Download this file and open it in any browser to use the map.
